# Mount Drive & chọn GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi   # kiểm tra A100

Mounted at /content/drive
Wed Aug 19 05:33:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

# Giải nén dataset

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Datasets/CAERS_train.zip"

!mkdir -p /content/CAER-S
!unzip -q "$DRIVE_ZIP_PATH" -d /content/CAER-S
!ls /content/CAER-S/train   # phải thấy 7 folder: Angry Disgust Fear Happy Neutral Sad Surprise

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise


# Clone repo

In [ ]:
!git clone https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 335, done.
remote: Counting objects: 100% (335/335), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 335 (delta 257), reused 324 (delta 249), pack-reused 0 (from 0)
Receiving objects: 100% (335/335), 30.25 MiB | 34.04 MiB/s, done.
Resolving deltas: 100% (257/257), done.
/content/Exp-CLIP


## Pull code

In [ ]:
%cd /content/Exp-CLIP
!git pull

# Cài dependencies

 Tạo venv Python 3.10

In [ ]:
!sudo apt-get -qq update > /dev/null   # làm mới danh sách gói, tránh lỗi 404 khi deadsnakes đã thay bản mới
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip

Cài torch (khớp CUDA) + dependencies + lavis vào venv đó

In [ ]:
!/content/venv310/bin/pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q salesforce-lavis
!/content/venv310/bin/pip install -q batch-face   # RetinaFace để crop mặt
# phải cài SAU lavis/batch-face: chúng kéo theo numpy 2.x và opencv 5.x (đòi numpy>=2); opencv 4.10 chạy được với numpy 1.x
!/content/venv310/bin/pip install -q "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, cv2, torch, lavis; from batch_face import RetinaFace; print('numpy', numpy.__version__, '| cv2', cv2.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

# Crop mặt CAER-S bằng RetinaFace
Tác giả crop mặt CAER-S bằng RetinaFace trước khi train (paper: ~45K ảnh mặt).

In [ ]:
# Crop mặt bằng RetinaFace (giữ mặt lớn nhất, bỏ ảnh không có mặt).
# Nếu đã có bản crop trên Drive thì chỉ giải nén, không crop lại.
import os
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"

if os.path.exists(FACE_ZIP):
    !mkdir -p /content/CAER-S_Face
    !unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
else:
    # chạy lại được nếu bị ngắt: ảnh đã crop sẽ được bỏ qua
    !/content/venv310/bin/python tools/crop_faces.py --src /content/CAER-S/train --dst /content/CAER-S_Face/train

Kiểm tra kết quả crop

In [ ]:
# Đếm số ảnh (paper: ~45K) và xem thử vài ảnh đã crop
import glob, random
import matplotlib.pyplot as plt
from PIL import Image

files = glob.glob("/content/CAER-S_Face/train/*/*")
print("Số ảnh sau khi crop:", len(files))
for d in sorted(glob.glob("/content/CAER-S_Face/train/*")):
    print(f"  {os.path.basename(d)}: {len(os.listdir(d))}")

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax, f in zip(axes.flat, random.sample(files, 16)):
    ax.imshow(Image.open(f).convert("RGB"))
    ax.set_title(f.split("/")[-2], fontsize=8)
    ax.axis("off")
plt.show()

Lưu bản crop lên Drive để lần sau không phải crop lại

In [ ]:
# Chỉ chạy khi crop đã xong và ảnh trông ổn
if not os.path.exists(FACE_ZIP):
    !cd /content/CAER-S_Face && zip -qr "$FACE_ZIP" train
!ls -lh "$FACE_ZIP"

In [ ]:
!du -sh /content/* 2>/dev/null | sort -rh | head -15
!du -sh /root/.cache/* ~/.cache/* 2>/dev/null | sort -rh | head -15
!du -sh /usr/local/lib/python3.10/dist-packages/* 2>/dev/null | sort -rh | head -15
!du -sh /content/venv310/lib/python3.10/site-packages/* 2>/dev/null | sort -rh | head -15

9.9G	/content/drive
9.2G	/content/CAER-S
5.7G	/content/venv310
68M	/content/Exp-CLIP
43G	/root/.cache/huggingface
890M	/root/.cache/clip
56M	/root/.cache/node-gyp
36K	/root/.cache/matplotlib
8.0K	/root/.cache/antigravity
4.0K	/root/.cache/torch
3.9G	/content/venv310/lib/python3.10/site-packages/torch
185M	/content/venv310/lib/python3.10/site-packages/triton
151M	/content/venv310/lib/python3.10/site-packages/pyarrow
112M	/content/venv310/lib/python3.10/site-packages/spacy
109M	/content/venv310/lib/python3.10/site-packages/scipy
101M	/content/venv310/lib/python3.10/site-packages/pycocoevalcap
69M	/content/venv310/lib/python3.10/site-packages/cv2
66M	/content/venv310/lib/python3.10/site-packages/plotly
65M	/content/venv310/lib/python3.10/site-packages/pandas
64M	/content/venv310/lib/python3.10/site-packages/cmake
61M	/content/venv310/lib/python3.10/site-packages/opencv_python_headless.libs
57M	/content/venv310/lib/python3.10/site-packages/sympy
53M	/content/venv310/lib/python3.10/site-pac

In [ ]:
!nproc

12


# Chạy train (đúng cấu hình runner.sh / paper, backbone L14, 3 seed)
Checkpoint, file resume và log được copy lên `OUT_DIR` sau mỗi epoch.

In [ ]:
import os
INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output"
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"

def job_id(seed):
    return f"expclip_l14_face_seed{seed}"

## seed 1

In [ ]:
SEED = 1
JOB_ID = job_id(SEED)
# nếu FlanT5 đã có trong cache (từ lần train trước trong cùng runtime) thì không gọi HuggingFace nữa
os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed $SEED \
  --job-id $JOB_ID \
  --instruction "$INSTRUCTION" \
  --load-model "CLIP_L14" \
  --train-data $TRAIN_DATA \
  --extra-checkpoint-dir $OUT_DIR

## seed 2

In [ ]:
SEED = 2
JOB_ID = job_id(SEED)
# nếu FlanT5 đã có trong cache (từ lần train trước trong cùng runtime) thì không gọi HuggingFace nữa
os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed $SEED \
  --job-id $JOB_ID \
  --instruction "$INSTRUCTION" \
  --load-model "CLIP_L14" \
  --train-data $TRAIN_DATA \
  --extra-checkpoint-dir $OUT_DIR

## seed 3

In [ ]:
SEED = 3
JOB_ID = job_id(SEED)
# nếu FlanT5 đã có trong cache (từ lần train trước trong cùng runtime) thì không gọi HuggingFace nữa
os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed $SEED \
  --job-id $JOB_ID \
  --instruction "$INSTRUCTION" \
  --load-model "CLIP_L14" \
  --train-data $TRAIN_DATA \
  --extra-checkpoint-dir $OUT_DIR

## Resume (chỉ dùng khi Colab bị ngắt giữa chừng)
Đặt `SEED` đúng seed đang train dở, chạy lại các cell setup phía trên (mount, giải nén, clone, venv, CAER-S_Face, config) rồi chạy cell này.

In [ ]:
# SEED = 2
# JOB_ID = job_id(SEED)
# os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"
# !/content/venv310/bin/python train.py \
#   --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
#   --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed $SEED \
#   --job-id $JOB_ID \
#   --instruction "$INSTRUCTION" \
#   --load-model "CLIP_L14" \
#   --train-data $TRAIN_DATA \
#   --extra-checkpoint-dir $OUT_DIR \
#   --resume $OUT_DIR/$JOB_ID-resume.pth

# Xác nhận checkpoint đã lưu trên Drive

In [ ]:
!ls -lh "$OUT_DIR"

# Giải nén test set

In [ ]:
!mkdir -p /content/static_test_sets
!unzip -q /content/drive/MyDrive/Datasets/static_test_sets.zip -d /content/static_test_sets
!ls /content/static_test_sets
!ls /content/static_test_sets/RAFDB/test   # phải thấy 0_Happiness ... 6_Fear

AffectNet7  AffectNet8	FERPlus  RAFDB
0_Happiness  1_Sadness	2_Neutral  3_Anger  4_Surprise	5_Disgust  6_Fear


# Chạy test (3 seed)

In [ ]:
# Test cả 3 seed; nếu runtime đã reset thì lấy checkpoint từ Drive
!mkdir -p checkpoint log
for SEED in [1, 2, 3]:
    JOB_ID = job_id(SEED)
    if not os.path.exists(f"checkpoint/{JOB_ID}-model.pth"):
        !cp "$OUT_DIR/$JOB_ID-model.pth" checkpoint/
    !/content/venv310/bin/python test.py --load-model "CLIP_L14" --job-id $JOB_ID --data-root /content/static_test_sets 2>&1 | tee log/$JOB_ID-test.txt
    !cp log/$JOB_ID-test.txt "$OUT_DIR/"

# Trung bình 3 seed

In [ ]:
# Trung bình UAR/WAR qua 3 seed (định dạng giống bảng trong paper)
import re
import pandas as pd

rows = []
for SEED in [1, 2, 3]:
    prompt, dataset = None, None
    for line in open(f"log/{job_id(SEED)}-test.txt"):
        if "Prompt Type:" in line:
            prompt = line.split("Prompt Type:")[1].strip()
        elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
            dataset = line.strip().split()[-1]
        elif line.startswith("UAR/WAR:"):
            uar, war = map(float, line.split(":")[1].strip().split("/"))
            rows.append(dict(seed=SEED, prompt=prompt, dataset=dataset, UAR=uar, WAR=war))

df = pd.DataFrame(rows)
summary = df.groupby(["prompt", "dataset"], sort=False)[["UAR", "WAR"]].agg(["mean", "std"]).round(2)
summary.to_csv(f"{OUT_DIR}/expclip_l14_face_summary.csv")
summary